# RAG benchmark · comparing SDK system prompts with DeepEval

Load the **same** exported `deepeval-goldens.json` for every system-prompt
variant. The `cli-agent` SDK runs the target; an independently selected
DeepEval judge scores factual accuracy, retrieval grounding, uncertainty
calibration (only when actual retrieved evidence is available), and style
compliance **separately**. Do not rewrite Golden reference answers to match
any response style. No generator-script imports are needed.

## 1 · Configure target prompt variants and independent judge

`PROMPT_VARIANTS` supplies an optional **inline system prompt** and
separate **style criteria** for each variant. The null prompt uses the
SDK-configured system prompt. These examples replace the base system prompt;
your technical and security instructions remain managed by the SDK.

`MAX_CASES=3` is a small initial smoke test. Set it to `None` when ready
to compare on the full exported dataset.

In [ ]:
from pathlib import Path
import asyncio
import hashlib
import json
import os
import tempfile
import tomllib

WORKSPACE = Path.cwd().resolve()
GOLDENS_FILE = WORKSPACE / "rag-goldens" / "deepeval-goldens.json"
OUTPUT_DIR = WORKSPACE / "rag-goldens" / "evaluation"

CONFIG_FILE = None
ADMIN_CONFIG_FILE = None
APPROVE_TOOLS = ()
MAX_CASES = 3
SDK_TIMEOUT_SECONDS = 600
FRESH_SDK = False
CACHE_EPOCH = ""       # change after updating external RAG index / MCP tools

# Names must be unique, lowercase ASCII; each variant uses independent caches.
PROMPT_VARIANTS = {
    "baseline": {
        "system_prompt": None,  # use explicitly loaded cli-agent config
        "style_criteria": "Answer clearly and directly, without unnecessary filler.",
    },
    "concise": {
        "system_prompt": "Answer technical questions accurately and concisely. Preserve essential caveats.",
        "style_criteria": "Be concise and direct while preserving essential technical qualifications.",
    },
    "detailed": {
        "system_prompt": "Answer technical questions accurately with useful explanations. Preserve exceptions and uncertainty.",
        "style_criteria": "Give useful explanations and necessary context without repetitive filler.",
    },
}

JUDGE_PROVIDER = "openrouter"   # openrouter / openai / ollama
JUDGE_MODEL = "openai/gpt-4.1"
JUDGE_BASE_URL = None
THRESHOLD = 0.7
JUDGE_MAX_CONCURRENT = 4
FRESH_EVALUATION = False

## 2 · Load native DeepEval Goldens, configure target factory and judge

The dataset and judge are initialized once. The default `create_target()`
factory builds a **`cli_agent.sdk.deepeval_adapter.DeepEvalTarget`**:
this is a DeepEval-observed adapter around a fresh SDK `Agent` per query,
not the SDK `Agent` itself.

The benchmark needs only a small asynchronous `ask(question, *,
expected_output, cached_response)` interface returning `actual_output`
and optional verified `knowledge_trace`. Replace **only** `create_target()`
to test a different agent. The factory also provides a stable cache identity
so response checkpoints cannot be reused across different targets.

**Requires the SDK version with `with_system_prompt(text)`.**

In [ ]:
import re
from cli_agent.sdk import AgentBuilder
from cli_agent.settings.config import ensure_default_config_file
from cli_agent.settings.admin_config import default_admin_config_file
from deepeval.dataset import EvaluationDataset
from deepeval.models import OpenAIModel, OpenRouterModel, OllamaModel

dataset = EvaluationDataset()
dataset.add_goldens_from_json_file(str(GOLDENS_FILE))
if MAX_CASES is not None:
    dataset = EvaluationDataset(goldens=dataset.goldens[:MAX_CASES])
if not dataset.goldens:
    raise ValueError("No Goldens available")

if not PROMPT_VARIANTS or any(
    not re.fullmatch(r"[a-z][a-z0-9_-]{0,40}", key)
    for key in PROMPT_VARIANTS
):
    raise ValueError("Prompt variant names must be distinct lowercase identifiers")
for key, variant in PROMPT_VARIANTS.items():
    if not isinstance(variant, dict):
        raise TypeError(f"{key}: expected variant configuration dict")
    prompt = variant.get("system_prompt")
    criteria = variant.get("style_criteria")
    if prompt is not None and (not isinstance(prompt, str) or not prompt.strip()):
        raise ValueError(f"{key}: system_prompt must be text or None")
    if not isinstance(criteria, str) or not criteria.strip():
        raise ValueError(f"{key}: style_criteria must be a nonempty string")

project_config = Path(CONFIG_FILE) if CONFIG_FILE else ensure_default_config_file()
admin_config = Path(ADMIN_CONFIG_FILE) if ADMIN_CONFIG_FILE else None
if JUDGE_PROVIDER == "openrouter":
    if JUDGE_BASE_URL:
        raise ValueError("OpenRouter uses its fixed endpoint")
    judge = OpenRouterModel(model=JUDGE_MODEL, api_key=os.environ["OPENROUTER_API_KEY"])
elif JUDGE_PROVIDER == "openai":
    judge = OpenAIModel(
        model=JUDGE_MODEL, api_key=os.environ["OPENAI_API_KEY"],
        **({"base_url": JUDGE_BASE_URL} if JUDGE_BASE_URL else {}),
    )
elif JUDGE_PROVIDER == "ollama":
    judge = OllamaModel(
        model=JUDGE_MODEL,
        **({"base_url": JUDGE_BASE_URL} if JUDGE_BASE_URL else {}),
    )
else:
    raise ValueError(f"Unknown JUDGE_PROVIDER: {JUDGE_PROVIDER}")
print(f"{len(dataset.goldens)} goldens, {len(PROMPT_VARIANTS)} variants; judge={JUDGE_PROVIDER}/{JUDGE_MODEL}")

### Target factory

The default implementation returns `TargetSpec(target=DeepEvalTarget(...),
cache_identity=...)`. `DeepEvalTarget.ask()` accepts the reference answer
for observation, never as an input to the underlying agent. The optional
`knowledge_trace` is used only if verified and complete.

To benchmark another agent, replace `create_target()` with a factory
returning a `BenchmarkTarget`-compatible adapter and a unique, stable
cache identity for that agent's effective configuration. An agent that is
an async context manager returning `AgentResult`-like results can reuse
`DeepEvalTarget(lambda: my_agent_factory())`; other APIs need a thin
adapter implementing the same `ask()` signature. Never pass the Golden's
expected answer to the target itself.

In [ ]:
from dataclasses import dataclass
from typing import Any, Mapping, Protocol
from cli_agent.sdk.deepeval_adapter import DeepEvalTarget

class BenchmarkTarget(Protocol):
    """Minimal interface consumed by the evaluation loop.

    expected_output is a reference for DeepEval tracing ONLY and must
    never be forwarded to the agent answering the question.
    """
    async def ask(
        self, question: str, *, expected_output: str,
        cached_response: Mapping[str, Any] | None = None,
    ) -> dict[str, Any]: ...

@dataclass(frozen=True)
class TargetSpec:
    target: BenchmarkTarget
    cache_identity: str

def effective_config_hash(prompt_text: str | None) -> str:
    # SDK project configuration is explicit. Workspace project.toml is not
    # merged automatically by the SDK.
    config_path = Path(project_config).expanduser().resolve(strict=True)
    files = [config_path]
    policy = Path(admin_config) if admin_config else default_admin_config_file()
    if policy.is_file():
        files.append(policy)
    if prompt_text is None:
        config_values = tomllib.loads(config_path.read_text(encoding="utf-8"))
        filename = config_values.get("agent", {}).get("system_prompt")
        if filename:
            files.append(WORKSPACE / ".cli-agent" / filename)
    digest = hashlib.sha256()
    for file in files:
        file = Path(file).expanduser().resolve(strict=True)
        digest.update(str(file).encode())
        digest.update(file.read_bytes())
    digest.update(json.dumps({
        "prompt_text": prompt_text,
        "approvals": sorted(APPROVE_TOOLS),
        "cache_epoch": CACHE_EPOCH,
        "workspace": str(WORKSPACE),
    }, sort_keys=True).encode())
    return digest.hexdigest()

def create_target(variant_id: str, variant: dict[str, Any]) -> TargetSpec:
    """Factory boundary: change ONLY this function for a different agent.

    DeepEvalTarget is an observed adapter, not a cli-agent Agent instance.
    Alternative: return TargetSpec(
        target=DeepEvalTarget(lambda: MyOtherAsyncAgent(...)),
        cache_identity="other-agent-v1:" + stable_config_digest,
    )
    MyOtherAsyncAgent must be an async context manager whose ask(question)
    returns an object with .answer and optional .knowledge_trace. For targets
    that already expose BenchmarkTarget.ask(), return them directly instead.
    """
    prompt_text = variant["system_prompt"]
    builder = AgentBuilder().with_admin_config(path=admin_config)
    builder.with_project_config(
        path=project_config, workspace=WORKSPACE,
        workspace_access="none", response_format="text",
    )
    if APPROVE_TOOLS:
        allowed = frozenset(APPROVE_TOOLS)
        async def approve(name, arguments):
            return name in allowed
        builder.with_approval_callback(approve)
    if prompt_text is not None:
        builder.with_system_prompt(prompt_text)
    return TargetSpec(
        target=builder.build_deepeval_target(include_knowledge_content=True),
        cache_identity="cli-agent-sdk:" + effective_config_hash(prompt_text),
    )

## 3 · DeepEval runs every prompt variant's agent

For each variant, DeepEval's `evals_iterator()` controls observed SDK
`target.ask()` calls. The iterator only applies a cheap response-captured
metric; paid Judge calls occur later via cache-aware `evaluate()`.
The target receives the **question only**, not the reference answer.

Each variant has its own answer cache, keyed by the effective SDK settings,
inline prompt, variant ID and Golden. The cache deliberately does not
fingerprint external indexes; change `CACHE_EPOCH` when they change.

In [ ]:
from deepeval.metrics import BaseMetric
from deepeval.evaluate import AsyncConfig, CacheConfig, DisplayConfig

class AnswerCaptured(BaseMetric):
    threshold = 1.0
    async_mode = False
    def measure(self, test_case, *args, **kwargs):
        self.score = float(bool((test_case.actual_output or "").strip()))
        self.success = self.is_successful()
        return self.score
    async def a_measure(self, test_case, *args, **kwargs):
        return self.measure(test_case)
    @property
    def __name__(self):
        return "Target answer captured"

options = {
    "cache_config": CacheConfig(use_cache=not FRESH_EVALUATION, write_cache=True),
    "display_config": DisplayConfig(
        results_folder=str(OUTPUT_DIR / "deepeval"), inspect_after_run=False,
    ),
}
cache_dir = OUTPUT_DIR / "responses"
cache_dir.mkdir(parents=True, exist_ok=True)
all_responses = {}
for variant_id, variant in PROMPT_VARIANTS.items():
    target_spec = create_target(variant_id, variant)
    target = target_spec.target
    fingerprint = target_spec.cache_identity
    if not isinstance(fingerprint, str) or not fingerprint.strip():
        raise ValueError(f"{variant_id}: target cache_identity must be nonempty")
    responses = {}

    for golden in dataset.evals_iterator(
        metrics=[AnswerCaptured()],
        identifier=f"rag-{variant_id}-target",
        async_config=AsyncConfig(run_async=False),
        **options,
    ):
        golden_id = (golden.additional_metadata or {})["golden_id"]
        cache_key = hashlib.sha256(json.dumps(
            [variant_id, fingerprint, golden_id, golden.input, golden.expected_output],
            ensure_ascii=False,
        ).encode()).hexdigest()
        checkpoint = cache_dir / f"{cache_key}.json"
        cached = None
        if (checkpoint.is_file() and not checkpoint.is_symlink()
                and not FRESH_SDK and checkpoint.stat().st_size <= 16_000_000):
            record = json.loads(checkpoint.read_text(encoding="utf-8"))
            if (isinstance(record, dict)
                    and isinstance(record.get("actual_output"), str)
                    and record["actual_output"].strip()
                    and (record.get("knowledge_trace") is None
                         or isinstance(record["knowledge_trace"], dict))):
                cached = record
        async with asyncio.timeout(SDK_TIMEOUT_SECONDS):
            response = await target.ask(
                golden.input, expected_output=golden.expected_output,
                cached_response=cached,
            )
        if cached is None:
            tmp = None
            try:
                with tempfile.NamedTemporaryFile(
                    mode="w", encoding="utf-8", dir=cache_dir,
                    prefix=".target-", suffix=".tmp", delete=False,
                ) as handle:
                    tmp = Path(handle.name)
                    os.chmod(tmp, 0o600)
                    json.dump(response, handle, ensure_ascii=False)
                    handle.flush()
                    os.fsync(handle.fileno())
                os.replace(tmp, checkpoint)
            finally:
                if tmp is not None:
                    tmp.unlink(missing_ok=True)
        responses[golden_id] = response

    all_responses[variant_id] = responses
    print(f"{variant_id}: {len(responses)} answers")

## 4 · Compare three independent dimensions

1. **Factual quality:** Correctness versus the unchanged reference answer,
   plus answer relevancy. Same rubric for all prompt variants.
2. **Grounding and uncertainty:** Faithfulness and contextual retrieval
   metrics plus uncertainty calibration **only for cases with verified actual
   selected Knowledge text**. Unverifiable cases remain not assessed here.
3. **Style compliance:** Separate G-Eval per variant using the explicitly
   configured `style_criteria`. Different style rubrics are **not** directly
   comparable as a single absolute score, but show compliance with each goal.

DeepEval owns scoring, caching, concurrency and reporting. Changing the
judge does not change cached target responses.

In [ ]:
from deepeval import evaluate
from deepeval.metrics import (
    GEval, AnswerRelevancyMetric, FaithfulnessMetric,
    ContextualRecallMetric, ContextualPrecisionMetric,
)
from deepeval.test_case import LLMTestCase, SingleTurnParams

evaluation_results = {}
for variant_id, variant in PROMPT_VARIANTS.items():
    answer_cases, retrieval_cases = [], []
    responses = all_responses[variant_id]
    for golden in dataset.goldens:
        response = responses[golden.additional_metadata["golden_id"]]
        base = dict(
            input=golden.input, expected_output=golden.expected_output,
            actual_output=response["actual_output"],
        )
        answer_cases.append(LLMTestCase(**base))
        trace = response.get("knowledge_trace") or {}
        selected = trace.get("selected") or []
        if (trace.get("content_included") is True
                and trace.get("truncated") is not True
                and selected and all(
                    isinstance(hit, dict) and isinstance(hit.get("content"), str)
                    and hit["content"].strip() for hit in selected
                )):
            retrieval_cases.append(LLMTestCase(
                **base, retrieval_context=[hit["content"] for hit in selected],
            ))

    correctness = GEval(
        name="Answer correctness",
        criteria=(
            "Judge factual correctness and completeness against the fixed reference. "
            "Accept paraphrases, concise or detailed explanations equally when "
            "all necessary facts and caveats are preserved; penalize invented claims."
        ),
        evaluation_params=[
            SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT,
            SingleTurnParams.EXPECTED_OUTPUT,
        ],
        model=judge, threshold=THRESHOLD,
    )
    style = GEval(
        name="Style compliance",
        criteria=(
            "Evaluate whether the actual response follows the requested response "
            "style, without judging factual correctness. Style goal: "
            + variant["style_criteria"]
        ),
        evaluation_params=[
            SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT,
        ],
        model=judge, threshold=THRESHOLD,
    )
    answer_results = evaluate(
        test_cases=answer_cases,
        metrics=[
            correctness, AnswerRelevancyMetric(model=judge, threshold=THRESHOLD),
            style,
        ],
        identifier=f"rag-{variant_id}-answer",
        async_config=AsyncConfig(max_concurrent=JUDGE_MAX_CONCURRENT),
        **options,
    )
    retrieval_results = None
    if retrieval_cases:
        uncertainty = GEval(
            name="Uncertainty calibration",
            criteria=(
                "Assess response calibration using the actual retrieved passages "
                "and question. Penalize unsupported confident claims and unjustified "
                "certainty; reward clearly stated limitations where evidence is "
                "insufficient. Do NOT reward blanket refusal if the provided "
                "retrieval passages adequately support an answer. "
                "Use the reference answer only as a factual target, not as evidence "
                "that the agent actually retrieved the necessary details."
            ),
            evaluation_params=[
                SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT,
                SingleTurnParams.EXPECTED_OUTPUT, SingleTurnParams.RETRIEVAL_CONTEXT,
            ],
            model=judge, threshold=THRESHOLD,
        )
        retrieval_results = evaluate(
            test_cases=retrieval_cases,
            metrics=[
                FaithfulnessMetric(model=judge, threshold=THRESHOLD),
                ContextualRecallMetric(model=judge, threshold=THRESHOLD),
                ContextualPrecisionMetric(model=judge, threshold=THRESHOLD),
                uncertainty,
            ],
            identifier=f"rag-{variant_id}-retrieval",
            async_config=AsyncConfig(max_concurrent=JUDGE_MAX_CONCURRENT),
            **options,
        )
    evaluation_results[variant_id] = {
        "answer": answer_results,
        "retrieval": retrieval_results,
        "retrieval_case_count": len(retrieval_cases),
    }
    print(f"{variant_id}: {len(answer_cases)} answers, "
          f"{len(retrieval_cases)} with verified retrieval")

## 5 · Side-by-side score summary

Scores below are averages over successful DeepEval metric evaluations, **not**
accuracy guarantees. Always inspect reasons, missing values and individual
test runs before choosing a prompt. Grounding/uncertainty scores are absent
when there is no verified retrieval context, and therefore may cover
different subsets across variants.

In [ ]:
from collections import defaultdict
from IPython.display import display
import pandas as pd

summary_rows = []
for variant_id, outputs in evaluation_results.items():
    grouped = defaultdict(list)
    errors = defaultdict(int)
    for stage in ("answer", "retrieval"):
        result = outputs[stage]
        for test_result in (getattr(result, "test_results", []) if result else []):
            for metric in (test_result.metrics_data or []):
                # DeepEval's GEval exposes names with a trailing " [GEval]".
                # Normalize the display suffix, not the underlying scores.
                name = metric.name.removesuffix(" [GEval]")
                if metric.error or metric.score is None:
                    errors[name] += 1
                else:
                    grouped[name].append(float(metric.score))
    row = {
        "variant": variant_id,
        "answer_cases": len(dataset.goldens),
        "retrieval_cases": outputs["retrieval_case_count"],
    }
    for name in ("Answer correctness", "Answer Relevancy", "Style compliance",
                 "Faithfulness", "Contextual Recall", "Contextual Precision",
                 "Uncertainty calibration"):
        values = grouped.get(name, [])
        row[name] = round(sum(values) / len(values), 3) if values else None
        row[name + " evaluated"] = len(values)
        row[name + " errors"] = errors.get(name, 0)
    summary_rows.append(row)
display(pd.DataFrame(summary_rows))
print("DeepEval reports:", OUTPUT_DIR / "deepeval")

### Interpretation

- **Correctness** should be compared on identical Goldens and judge settings.
- **Style** is evaluated against a *variant-specific* criterion; do not rank
  raw style numbers across dissimilar criteria without a shared rubric.
- **Grounding and uncertainty** are only measured where full actual selected
  Knowledge passages are available. Missing retrieval is **not** scored as
  success or failure and is visible via the case counts.
- **No counterfactual guarantee:** the judge sees selected passages, not all
  evidence that could have been retrieved. "Uncertainty calibration" is an
  LLM-judged diagnostic, not a proof of warranted abstention.
- Source reference answers remain unchanged for all variants. Responding
  cautiously when facts were not retrieved may lower completeness; inspect
  correctness and uncertainty jointly rather than combining them blindly.
- Change `CACHE_EPOCH` or set `FRESH_SDK=True` when the indexed corpus or
  tool environment changes. Private caches contain potentially sensitive
  selected passages.
- To evaluate a different agent, change `create_target()`, not the
  DeepEval loop. Use a distinct `cache_identity` for each agent configuration.
